### DATA PROCESSING NOTEBOOK

In [10]:
# Import necessary libraries
import pandas as pd
import os
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import joblib

In [11]:
# Import data
dir_path = os.path.join(os.path.dirname(os.getcwd()), "data", "processed")
data_path = os.path.join(dir_path, "interpolated_data.xlsx")
df = pd.read_excel(data_path)
df.shape

(400, 4)

In [12]:
df.head(10)

,1/T,ln strain rate,Strain,σ/σmax
0,0.000756,-2.302585,0.050000,0.270714
1,0.000756,-2.302585,0.072917,0.283466
2,0.000756,-2.302585,0.095833,0.292302
3,0.000756,-2.302585,0.118750,0.299277
4,0.000756,-2.302585,0.141667,0.306447
5,0.000756,-2.302585,0.164583,0.315667
6,0.000756,-2.302585,0.187500,0.326199
7,0.000756,-2.302585,0.210417,0.335541
8,0.000756,-2.302585,0.233333,0.342615
9,0.000756,-2.302585,0.256250,0.347968


In [13]:
df.describe()

,1/T,ln strain rate,Strain,σ/σmax
count,400.000000,400.000000,400.000000,400.000000
mean,0.000803,0.677013,0.325000,0.661755
std,0.000036,2.008999,0.165461,0.172150
min,0.000756,-2.302585,0.050000,0.270714
25%,0.000778,-0.575646,0.187500,0.534763
50%,0.000802,1.151293,0.325000,0.662469
75%,0.000826,2.403951,0.462500,0.781940
max,0.000853,2.708050,0.600000,1.000000


In [14]:
df.isnull().sum()

1/T               0
ln strain rate    0
Strain            0
σ/σmax            0
dtype: int64

### DATA SPLITTING

In [15]:
# Splitting the data
X = df.drop(["σ/σmax"], axis = 1)
y = df["σ/σmax"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,shuffle=True)

# Further split it to validation as well
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.15, random_state=42)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_train: {y_test.shape}")

X_train: (272, 3), y_train: (272,)
X_val: (48, 3), y_val: (48,)
X_test: (80, 3), y_train: (80,)


### FEATURE SCALING

In [16]:
# Scaling with min max scaler to follow the normal operating range
scaler = MinMaxScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


In [17]:
np.save('../data/processed/X_train_scaled.npy', X_train_scaled)
np.save('../data/processed/X_val_scaled.npy', X_val_scaled)
np.save('../data/processed/X_test_scaled.npy', X_test_scaled)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_val.npy', y_val)
np.save('../data/processed/y_test.npy', y_test)

print("All splits saved successfully")

All splits saved successfully


In [18]:
# Saving the scaler.
joblib.dump(scaler, '../artifacts/scaler.pkl')
print("Scaler saved successfully")
print(f"Feature range after scaling: {scaler.data_range_}")

Scaler saved successfully
Feature range after scaling: [9.66570203e-05 5.01063529e+00 5.50000000e-01]


## Preprocessing Conclusions

The preprocessing pipeline prepared the interpolated dataset for model training
in three steps.

**Feature Selection**
The redundant column ε' was dropped, leaving three independent input features:
1/T, ln strain rate, and Strain, with σ/σmax as the target variable.

**Data Splitting**
The dataset was split into three sets using a random stratified shuffle:
- Training set (68%) → model learning
- Validation set (12%) → performance monitoring during training
- Test set (20%) → final honest evaluation

**Feature Scaling**
MinMaxScaler was fitted exclusively on the training set and applied to all
three splits, transforming all input features to the [0, 1] range. This is
consistent with the target variable σ/σmax which is naturally bounded in
the same range. The fitted scaler was saved to artifacts/scaler.pkl for
use at inference time.

All processed splits were saved to data/processed/ as .npy files and are
ready for model training.